In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

In [2]:
# Step 1: Load datasets for multiple languages

datasets = {
    "English": load_dataset("universal_dependencies", "en_ewt")["train"],
    "Hindi": load_dataset("universal_dependencies", "hi_hdtb")["train"],
    "German": load_dataset("universal_dependencies", "de_gsd")["train"],
    "Spanish": load_dataset("universal_dependencies", "es_gsd")["train"]
}

In [3]:
# Step 2: Load pretrained BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Step 2: Filter clean sentences

def get_clean_data(data, limit=200):
    clean_data = []

    for example in data.select(range(limit)):
        tokens = example["tokens"]

        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt"
        )

        bert_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        if len(bert_tokens) - 2 >= len(tokens):
            clean_data.append(example)

    return clean_data

In [5]:
# Step 3: Run experiment for one dataset

def run_experiment(data, max_sentences=30):

    clean_data = get_clean_data(data)
    subset = clean_data[:max_sentences]

    all_results = []

    for example in subset:
        tokens = example["tokens"]
        heads = example["head"]

        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt"
        )

        outputs = model(**inputs)
        attentions = outputs.attentions

        for layer in range(len(attentions)):
            for head in range(attentions[layer].shape[1]):

                attn = attentions[layer][0][head]
                attn = attn[1:-1, 1:-1]

                pred_row = torch.argmax(attn, dim=1).tolist()
                pred_col = torch.argmax(attn, dim=0).tolist()

                acc_row = compute_accuracy(pred_row, heads)
                acc_col = compute_accuracy(pred_col, heads)

                all_results.append({
                    "layer": layer,
                    "head": head,
                    "score": max(acc_row, acc_col)
                })

    # aggregate
    agg = defaultdict(list)

    for r in all_results:
        key = (r["layer"], r["head"])
        agg[key].append(r["score"])

    final_scores = []

    for (layer, head), scores in agg.items():
        final_scores.append({
            "layer": layer,
            "head": head,
            "avg_acc": np.mean(scores)
        })

    return pd.DataFrame(final_scores)

In [ ]:
#Define accuracy computation

def compute_accuracy(pred_heads, heads):
    correct = 0
    total = 0

    for i in range(len(heads)):
        if int(heads[i]) == 0:
            continue  # skip ROOT

        gold = int(heads[i]) - 1
        pred = pred_heads[i]

        total += 1
        if gold == pred:
            correct += 1

    return correct / total if total > 0 else 0

In [7]:
# Step 4: Run experiment for all languages

results = {}

for lang, data in datasets.items():
    print(f"Running for {lang}...")
    results[lang] = run_experiment(data)

Running for English...


ValueError: invalid literal for int() with base 10: 'None'

In [ ]:
for lang in results:
    print(lang, results[lang].shape)

In [ ]:
# Step 5: Top heads per language

for lang in results:
    print(f"\nTop heads for {lang}:")
    print(results[lang].sort_values(by="avg_acc", ascending=False).head(5))

In [ ]:
# Step 6: Summary statistics

for lang in results:
    df = results[lang]
    print(f"\n{lang}:")
    print("Max:", df["avg_acc"].max())
    print("Mean:", df["avg_acc"].mean())
    print("Min:", df["avg_acc"].min())

In [ ]:
# Step 7: Heatmaps

for lang in results:
    df = results[lang]
    
    heatmap = np.zeros((12, 12))
    
    for _, row in df.iterrows():
        heatmap[int(row["layer"]), int(row["head"])] = row["avg_acc"]

    plt.figure(figsize=(6,5))
    plt.imshow(heatmap, cmap='viridis', aspect='auto')
    plt.colorbar(label="Accuracy")

    plt.title(f"{lang}: Dependency Alignment")
    plt.xlabel("Head")
    plt.ylabel("Layer")

    plt.show()

In [ ]:
# Step 8: Distribution plots

for lang in results:
    df = results[lang]
    
    plt.figure()
    plt.hist(df["avg_acc"], bins=10)
    
    plt.title(f"{lang}: Accuracy Distribution")
    plt.xlabel("Accuracy")
    plt.ylabel("Number of Heads")
    
    plt.show()